# Beyond A/B Testing

**Philosophy:** Deliberately short. This note is a decision framework and concept map — not a full implementation guide. When you can't randomize, use this to pick the right method. Full implementations live in `causal_ml/`.

---

## Decision Table

| If you need to... | Go to |
| :--- | :--- |
| Understand why A/B isn't always possible | §1 — Why You Can't Always Randomize |
| Pick the right observational method | §2 — Decision Framework |
| Compare methods at a glance | §3 — Method Comparison Table |

---
## §1 — Why You Can't Always Randomize

Randomized A/B testing is the gold standard — but it requires conditions that are often not met in practice.

| Reason | Example | Consequence |
| :--- | :--- | :--- |
| **Ethical constraints** | Can't randomly deny medical treatment or financial credit | Must use observational data |
| **One-sided markets** | Pricing or fee changes affect all users — can't split | Geo or time-based design needed |
| **Small population** | B2B product with 200 enterprise users | Insufficient power for standard A/B |
| **Historical data only** | Evaluating a past policy that already ran | No future randomization possible |
| **Platform-level changes** | Infrastructure change — can't toggle per user | Switchback or pre/post design |
| **Network effects** | Social feature — users interact, contaminating groups | Cluster randomization or observational |
| **Long time horizon** | Measuring 1-year LTV impact | Can't wait; use proxy or observational |

**The core problem:**
Without randomization, the treated and untreated groups differ systematically before treatment. Naive comparison mixes the treatment effect with pre-existing differences (confounding). Every observational method is an attempt to adjust for this — with different assumptions, data requirements, and validity.

---
## §2 — Decision Framework

```
Can you randomly assign treatment?
│
├── YES ──────────────────────────────────────────────────────────────────────
│   └── A/B Test (see Exp_Design.ipynb, Exp_Analysis.ipynb)
│
└── NO — choose based on what data structure you have:
    │
    ├── Is there a sharp cutoff / threshold that determines treatment?
    │   (score above 700 gets a loan, age ≥ 65 gets a benefit)
    │   └── Regression Discontinuity (RD)
    │       Assumption: units just above/below the cutoff are comparable
    │       Strength: near-experimental validity near the cutoff
    │       Limitation: only identifies the effect at the cutoff, not population ATE
    │
    ├── Do you have pre/post data AND a control group that was never treated?
    │   (policy changed in some states but not others, some stores got a renovation)
    │   └── Difference-in-Differences (DiD)
    │       Assumption: parallel trends — treated and control would have
    │                   followed the same trend without treatment
    │       Strength: controls for time-invariant confounders
    │       Limitation: parallel trends is untestable; fails if groups diverge pre-treatment
    │
    ├── Do you have a variable that affects treatment but not outcome directly?
    │   (lottery assignment, distance to a facility, policy eligibility cutoff)
    │   └── Instrumental Variables (IV)
    │       Assumption: instrument is relevant (correlated with treatment) and
    │                   exogenous (uncorrelated with outcome except through treatment)
    │       Strength: handles unmeasured confounders
    │       Limitation: valid instruments are rare; identifies LATE, not ATE
    │
    ├── Do you have rich pre-treatment covariates and assume no unmeasured confounders?
    │   (observational data with demographics, history, behavior)
    │   └── Matching / Propensity Score Methods (PSM, IPW, Doubly Robust)
    │       Assumption: unconfoundedness — no unmeasured confounders
    │                   (selection on observables)
    │       Strength: flexible, works on any observational data
    │       Limitation: cannot handle unmeasured confounders
    │
    └── Do you want individual-level treatment effects, not just average?
        └── CATE / Uplift Modeling (causal forests, X-learner, T-learner)
            Assumption: unconfoundedness (same as matching)
            Strength: heterogeneous treatment effects by subgroup
            Limitation: harder to validate; assumes rich covariate data
```

---
## §3 — Method Comparison Table

| Method | Key assumption | Data required | Identifies | Handles unmeasured confounders? | Go deeper |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **A/B Test** | Random assignment | RCT data | ATE | ✅ Yes — by design | `Exp_Design`, `Exp_Analysis` |
| **Diff-in-Diff** | Parallel trends | Panel data, pre+post, control group | ATT | ✅ For time-invariant only | `causal_ml/` |
| **Regression Discontinuity** | Continuity at cutoff | Running variable + cutoff | LATE (at cutoff) | ✅ Near cutoff | `causal_ml/` |
| **Instrumental Variables** | Valid instrument | Instrument + treatment + outcome | LATE (compliers) | ✅ Yes | `causal_ml/` |
| **Matching / PSM / IPW** | Unconfoundedness | Rich covariates | ATE / ATT | ❌ No | `causal_ml/` |
| **CATE / Uplift** | Unconfoundedness | Rich covariates + many observations | CATE (individual) | ❌ No | `causal_ml/` |

**ATE** = Average Treatment Effect (population average)
**ATT** = Average Treatment Effect on the Treated
**LATE** = Local Average Treatment Effect (only for a subset — near cutoff or compliers)
**CATE** = Conditional Average Treatment Effect (individual-level, conditional on covariates)

---

## Key Questions to Ask in an Interview

When asked *"how would you measure the effect of X without an A/B test?"*, structure your answer around:

```
1. Why can't we randomize?         Identify the specific constraint.

2. What's the causal question?     What is the treatment? What is the outcome?
                                   What would we see in the counterfactual world?

3. What data do we have?           Panel? Cross-section? Pre/post? Instrument?

4. What's the key assumption?      Name it explicitly — parallel trends, unconfoundedness,
                                   valid instrument. Don't skip this.

5. How would we validate it?       Pre-trend test for DiD. Placebo test for RD.
                                   First-stage F-stat for IV. Balance check for matching.

6. What does it identify?          ATE? ATT? LATE? Be precise — it affects generalizability.
```

**Common mistakes:**
- Jumping to PSM without acknowledging the unconfoundedness assumption — interviewers always probe this
- Claiming DiD identifies ATE when it only identifies ATT (effect on the treated group, not the general population)
- Forgetting to validate the key assumption — every observational method has one critical assumption that must be tested or at least discussed
- Treating observational estimates with the same confidence as A/B results — always caveat with the assumption and its limitations